In [ ]:
# Importations
import os
import sys
import logging
import requests
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType, DoubleType
)
from pyspark.sql.functions import current_timestamp


In [ ]:
# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)


In [ ]:
# Chargement des variables d'environnement (.env)
load_dotenv()


In [ ]:
# Paramètres de connexion PostgreSQL
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}


In [ ]:
# Coordonnées des 3 régions avec leurs noms
REGIONS = {
    "Region A": {
        "latitude": 34.0522, 
        "longitude": -118.2437, 
        "region_name": "Los Angeles, California, USA"
    },
    "Region B": {
        "latitude": 36.7783, 
        "longitude": -119.4179, 
        "region_name": "Fresno/Central Valley, California, USA"
    },
    "Region C": {
        "latitude": 40.7128, 
        "longitude": -74.006, 
        "region_name": "New York City, New York, USA"
    }
}

# Paramètres API Open-Meteo
OPENMETEO_PARAMS = {
    "hourly": [
        "wind_speed_100m",       # Vitesse à hauteur de hub (~100m)
        "wind_gusts_10m",        # Rafales (impact maintenance)
        "temperature_2m",        # Température (densité de l'air)
        "pressure_msl",          # Pression (calcul densité air)
        "precipitation"          # Précipitations (impact opérationnel)
    ]
}


In [ ]:
# Date cible pour l'ingestion
target_date = "2024-06-16"  # Format: YYYY-MM-DD
print(f"Date cible : {target_date}")


In [ ]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WeatherAPIBronzeIngestion") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()


In [ ]:
# Récupération et traitement des données pour toutes les régions
all_records = []

for region, coords in REGIONS.items():
    # Appel API Open-Meteo
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": coords["latitude"],
        "longitude": coords["longitude"],
        "start_date": target_date,
        "end_date": target_date,
        "hourly": ",".join(OPENMETEO_PARAMS["hourly"]),
        "timezone": "UTC"
    }
    
    try:
        logger.info(f"Appel API pour {region} ({coords['latitude']}, {coords['longitude']}) le {target_date}")
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        api_result = {"success": True, "data": response.json()}
    except requests.RequestException as e:
        logger.error(f"Erreur API pour {region}: {e}")
        continue
    
    # Transformation des données horaires en intervalles de 10 minutes
    data = api_result["data"]
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    
    # Extraction des données horaires
    hourly_data = {
        "wind_speed_100m": hourly.get("wind_speed_100m", []),
        "wind_gusts_10m": hourly.get("wind_gusts_10m", []),
        "temperature_2m": hourly.get("temperature_2m", []),
        "pressure_msl": hourly.get("pressure_msl", []),
        "precipitation": hourly.get("precipitation", [])
    }
    
    # Filtrer uniquement la date cible
    hourly_filtered = {}
    for i, timestamp_str in enumerate(times):
        if timestamp_str.startswith(target_date):
            hour = int(timestamp_str.split("T")[1].split(":")[0])
            hourly_filtered[hour] = {
                key: values[i] for key, values in hourly_data.items()
            }
    
    if not hourly_filtered:
        logger.warning(f"Aucune donnée pour {region} le {target_date}")
        continue
    
    # Générer des enregistrements toutes les 10 minutes de 00:00 à 23:50
    for hour in range(24):
        for minute in range(0, 60, 10):
            # Interpolation entre l'heure actuelle et l'heure suivante
            current_hour_data = hourly_filtered.get(hour, {})
            next_hour = (hour + 1) % 24
            next_hour_data = hourly_filtered.get(next_hour, {})
            
            # Fraction pour l'interpolation (0.0 à 0.833 pour 0min à 50min)
            fraction = minute / 60.0
            
            # Construction du timestamp
            time_str = f"{hour:02d}:{minute:02d}:00"
            
            # Fonction d'interpolation inline
            def interpolate(val1, val2):
                if val1 is None or val2 is None:
                    return val1 if val1 is not None else val2
                return val1 + (val2 - val1) * fraction
            
            record = {
                "date": target_date,
                "time": time_str,
                "latitude": data["latitude"],
                "longitude": data["longitude"],
                "region": region,
                "region_name": coords["region_name"],
                "wind_speed_100m": interpolate(
                    current_hour_data.get("wind_speed_100m"),
                    next_hour_data.get("wind_speed_100m")
                ),
                "wind_gusts_10m": interpolate(
                    current_hour_data.get("wind_gusts_10m"),
                    next_hour_data.get("wind_gusts_10m")
                ),
                "temperature_2m": interpolate(
                    current_hour_data.get("temperature_2m"),
                    next_hour_data.get("temperature_2m")
                ),
                "pressure_msl": interpolate(
                    current_hour_data.get("pressure_msl"),
                    next_hour_data.get("pressure_msl")
                ),
                "precipitation": interpolate(
                    current_hour_data.get("precipitation"),
                    next_hour_data.get("precipitation")
                )
            }
            all_records.append(record)
    
    logger.info(f"{len(hourly_filtered) * 6} enregistrements créés pour {region}")

print(f"Total d'enregistrements récupérés : {len(all_records)}")


In [ ]:
# Création du DataFrame Spark
schema = StructType([
    StructField("date", StringType(), False),
    StructField("time", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("region", StringType(), False),
    StructField("region_name", StringType(), False),
    StructField("wind_speed_100m", FloatType(), True),
    StructField("wind_gusts_10m", FloatType(), True),
    StructField("temperature_2m", FloatType(), True),
    StructField("pressure_msl", FloatType(), True),
    StructField("precipitation", FloatType(), True)
])

df_weather = spark.createDataFrame(all_records, schema) \
    .withColumn("ingested_at", current_timestamp())

# Affichage des premières lignes
df_weather.show(10)
print(f"Nombre total de lignes : {df_weather.count()}")


In [ ]:
# Création du schéma et de la table dans PostgreSQL (si besoin)
try:
    conn = psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    with conn:
        with conn.cursor() as cur:
            cur.execute("CREATE SCHEMA IF NOT EXISTS bronze")
            cur.execute("DROP TABLE IF EXISTS bronze.weatherforecastapi_raw")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS bronze.weatherforecastapi_raw (
                    date DATE NOT NULL,
                    time TIME NOT NULL,
                    latitude NUMERIC(9,6) NOT NULL,
                    longitude NUMERIC(9,6) NOT NULL,
                    region VARCHAR(100) NOT NULL,
                    region_name VARCHAR(255) NOT NULL,
                    wind_speed_100m NUMERIC(6,2),
                    wind_gusts_10m NUMERIC(6,2),
                    temperature_2m NUMERIC(5,2),
                    pressure_msl NUMERIC(7,2),
                    precipitation NUMERIC(8,2),
                    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    source_api VARCHAR(100) DEFAULT 'Open-Meteo',
                    UNIQUE(region, date, time)
                )
            """)
            cur.execute("""
                CREATE INDEX IF NOT EXISTS idx_weather_region_datetime 
                ON bronze.weatherforecastapi_raw(region, date, time)
            """)
            cur.execute("""
                CREATE INDEX IF NOT EXISTS idx_weather_coordinates 
                ON bronze.weatherforecastapi_raw(latitude, longitude)
            """)
            conn.commit()
    print("Schéma et table créés avec succès.")
except Exception as e:
    print(f"Erreur lors de la création du schéma/table : {e}")
    raise


In [ ]:
# Insertion des données dans PostgreSQL via JDBC (triées par date, time, region)
df_weather_sorted = df_weather.orderBy("date", "time", "region")

df_weather_sorted.write \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "bronze.weatherforecastapi_raw") \
    .option("stringtype", "unspecified") \
    .options(**JDBC_PROPS) \
    .mode("append") \
    .save()

print(f"{df_weather_sorted.count()} lignes insérées dans bronze.weatherforecastapi_raw (ordonnées par date, time, region)")


In [ ]:
# Arrêt de la session Spark
spark.stop()
print("Session Spark fermée.")
